# 01 — Explore Dataset

This notebook inspects the preprocessed Track 2 neural recordings from the Mouse vs AI benchmark. Track 2 evaluates how well model representations predict single-neuron activity recorded from mouse visual cortex during a virtual foraging task. This notebook lets you examine those recordings directly — no model evaluation or regression fitting is performed.

**What this notebook does:**
- Loads all sessions in the selected set using the same loader as `02_track2_evaluation.ipynb`
- Prints a compact overview table across sessions
- Visualizes stimulus frames, neural activity, and modality alignment for one selected session

**What this notebook does NOT do:**
- Load or evaluate any model
- Fit any regression
- Require a GPU

Run all cells top to bottom. Expected runtime: a few seconds.

## Data download

The preprocessed Track 2 dataset can be downloaded from **[https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/YB8J31](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/YB8J31)**. After downloading, copy or symlink the folder to `mouse_vs_ai_data_preprocessed/` (one level above the repo root) so that the paths match those used in `02_track2_evaluation.ipynb`.

For completeness, the raw (unprocessed) recording data can be downloaded from **[INSERT LINK]**.

In [ ]:
# ── User Settings ─────────────────────────────────────────────────────────────────────────────
from pathlib import Path
import sys

# Which session set to load — same options as 02_track2_evaluation.ipynb:
#   "official_3session"  →  3 sessions used for the official Track 2 leaderboard
#   "full_9session"      →  all 9 released sessions (3 original + 6 additional)
SESSION_SET = "official_3session"

# Which session to use for the detail view and visualizations.
# Must be one of the stems in the selected SESSION_SET above.
VIZ_SESSION = "tigre569_p2s38_mousevAI_perturbs_preprocessed"

# Paths — same convention as 02_track2_evaluation.ipynb
ROOT     = Path().resolve()
DATA_DIR = Path("./data")
SAVE_DIR = ROOT / "results" / "explore"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Time window for detail visualizations (frame indices into VIZ_SESSION valid frames)
VIZ_START = 100
VIZ_LEN   = 300   # 300 frames ≈ 30 s at 10 Hz

sys.path.insert(0, str(ROOT))

In [ ]:
# ── Imports and session index ───────────────────────────────────────────────────────────
import json
import numpy as np
import matplotlib.pyplot as plt

from src.data_loading import load_all_bundles, load_session_bundle

# Session stems — same dict as in 02_track2_evaluation.ipynb
_SESSION_STEMS = {
    "official_3session": [
        "tigre569_p2s38_mousevAI_perturbs_preprocessed",
        "tigre613_p2s23_mousevAI_perturbs_preprocessed",
        "tigre847_p2s8_mousevAI_perturbs_preprocessed",
    ],
    "full_9session": [
        "tigre569_p2s38_mousevAI_perturbs_preprocessed",
        "tigre613_p2s23_mousevAI_perturbs_preprocessed",
        "tigre847_p2s8_mousevAI_perturbs_preprocessed",
        "tigre569_p2s28_mousevAI_perturbs_preprocessed",
        "tigre569_p2s35_mousevAI_perturbs_preprocessed",
        "tigre613_p2s21_mousevAI_perturbs_preprocessed",
        "tigre613_p2s22_mousevAI_perturbs_preprocessed",
        "tigre847_p2s5_mousevAI_perturbs_preprocessed",
        "tigre847_p2s9_mousevAI_perturbs_preprocessed",
    ],
}

if SESSION_SET not in _SESSION_STEMS:
    raise ValueError(
        f"SESSION_SET must be 'official_3session' or 'full_9session', got {SESSION_SET!r}"
    )
if VIZ_SESSION not in _SESSION_STEMS[SESSION_SET]:
    raise ValueError(
        f"VIZ_SESSION '{VIZ_SESSION}' is not in SESSION_SET '{SESSION_SET}'.\n"
        f"Available stems: {_SESSION_STEMS[SESSION_SET]}"
    )

stems         = _SESSION_STEMS[SESSION_SET]
session_paths = [str(DATA_DIR / f"{s}.npz") for s in stems]

missing = [p for p in session_paths if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} session file(s) not found under {DATA_DIR}.\n"
        "Check that DATA_DIR points to the released preprocessed dataset directory.\n"
        "Missing:\n" + "\n".join(f"  {p}" for p in missing)
    )

---
## Loading sessions

All sessions are loaded with `load_all_bundles` — the same function used by `02_track2_evaluation.ipynb`. It applies blackout masking automatically, discarding invalid frames (scene transitions) from each session. The result is a list of bundles, one per session, each containing `frames`, `spikes`, `T` (valid frame count), and `N` (neuron count).

The overview table below summarises all sessions in the selected set. After the table, one session (`VIZ_SESSION`) is selected for a closer look.

In [ ]:
# ── Load all sessions ──────────────────────────────────────────────────────────
print(f"Loading {SESSION_SET}  ({len(stems)} sessions) ...\n")
bundles = load_all_bundles(session_paths)

In [ ]:
# ── Session overview table ──────────────────────────────────────────────────────────────
# Computes from loaded bundles only — no sidecar files required.
W = 80
print(f"\n{'\u2500'*W}")
print(f"  {'Session':<52} {'Frames':>7} {'Neurons':>8}")
print(f"{'\u2500'*W}")

for b, stem in zip(bundles, stems):
    marker = " *" if stem == VIZ_SESSION else "  "
    print(f"{marker} {stem:<52} {b['T']:>7,} {b['N']:>8,}")

print(f"{'\u2500'*W}")
print(f"  Total neurons (all sessions): {sum(b['N'] for b in bundles):,}")
print(f"  Filter: C0.005  (FR ≥ 100 total spikes AND R_visual ≥ 0.005)")
print(f"\n  * = session selected for detail view (VIZ_SESSION)")

---
## Selected session — detail view

The cells below inspect `VIZ_SESSION` in detail. Change `VIZ_SESSION` in the User Settings cell and re-run from here to explore a different recording.

In [ ]:
# ── Load selected session ──────────────────────────────────────────────────────────────
# Reuse the already-loaded bundle — no second disk read needed.
viz_idx = stems.index(VIZ_SESSION)
bundle  = bundles[viz_idx]

npz_path = DATA_DIR / f"{VIZ_SESSION}.npz"

# Load arrays not returned by load_session_bundle
raw           = np.load(npz_path, allow_pickle=False)
time_full     = raw["time"]
cellarea      = raw["cellarea"][0]        # (N_kept,)
blackout_mask = raw["blackouts"][0] == 0  # (T_full,)
time_valid    = time_full[blackout_mask]  # (T_valid,)

# Clamp visualization window to available frames
if VIZ_START >= bundle["T"]:
    print(f"WARNING: VIZ_START={VIZ_START} >= T={bundle['T']}; resetting to 0.")
    VIZ_START = 0
VIZ_LEN = min(VIZ_LEN, bundle["T"] - VIZ_START)

# ── Print session details ────────────────────────────────────────────────────────
T_full = raw["frames"].shape[0]
dt     = float(np.median(np.diff(time_valid)))

print(f"{'\u2500'*58}")
print(f"  Session        : {bundle['mouse_name']}")
print(f"  Valid frames   : {bundle['T']:,}  "
      f"(total {T_full:,},  blackouts {(~blackout_mask).sum():,})")
print(f"  Frame shape    : {bundle['frames'].shape[1:]}  [C × H × W]")
print(f"  Neurons        : {bundle['N']:,}")
print(f"  Spike array    : {bundle['spikes'].shape}  [N × T]")
print(f"  Time range     : {time_valid[0]:.1f} s – {time_valid[-1]:.1f} s")
print(f"  Sampling rate  : ~{1/dt:.0f} Hz  ({dt*1000:.0f} ms / bin)")
print(f"  Viz window     : frames {VIZ_START}–{VIZ_START+VIZ_LEN-1} "
      f"({time_valid[VIZ_START]:.1f} s – {time_valid[VIZ_START+VIZ_LEN-1]:.1f} s)")
print(f"{'\u2500'*58}")

areas, counts = np.unique(cellarea, return_counts=True)
print(f"\nNeurons per brain area index:")
for a, c in zip(areas, counts):
    print(f"  area {int(a)}: {c:>5} neurons")

print(f"\nNeuron filter (C0.005 — applied at preprocessing):")
print(f"  FR criterion   : total spike count ≥ 100 over valid frames")
print(f"  R_visual crit. : Pearson r (image→spike linear prediction) ≥ 0.005")
print(f"  Neurons kept   : {bundle['N']:,}  (all neurons in this release pass C0.005)")

---
## Neuron filtering

The preprocessed sessions in this release contain only the subset of neurons used for Track 2 evaluation and are ready to use as-is — no further filtering is required. Two criteria must both be satisfied for a neuron to be retained:

- **FR criterion** — total spike count over valid frames ≥ 100 (removes near-silent neurons)
- **C0.005 criterion** — visual responsiveness R_visual ≥ 0.005, where R_visual is the Pearson correlation between a linearly predicted spike train (from pixel features via Ridge regression) and the observed spike train

Together these criteria retain roughly 18–39% of the recorded population (varies by session) — the most active and visually responsive neurons. This release contains only the filtered population; the full unfiltered population is not included.

---
## Example stimulus frames

The stimulus is a grayscale virtual foraging environment rendered at 86 × 155 pixels and sampled at ~10 Hz. Below are 9 evenly spaced frames from the visualization window.

In [ ]:
# ── Example frames ─────────────────────────────────────────────────────────────
N_FRAMES = 9
frame_indices = np.linspace(VIZ_START, VIZ_START + VIZ_LEN - 1, N_FRAMES, dtype=int)

fig, axes = plt.subplots(3, 3, figsize=(9, 5))
for ax, fi in zip(axes.flat, frame_indices):
    frame = bundle["frames"][fi, 0]   # (86, 155) float32
    ax.imshow(frame, cmap="gray", vmin=0, vmax=1, aspect="auto")
    ax.set_title(f"t = {time_valid[fi]:.1f} s", fontsize=8)
    ax.axis("off")

fig.suptitle(f"Stimulus frames — {bundle['mouse_name']}", fontsize=10)
fig.tight_layout()
fig.savefig(SAVE_DIR / "example_frames.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {SAVE_DIR / 'example_frames.png'}")

---
## Neural activity

Neural activity is stored as binned spike counts. The raster shows time bins with nonzero inferred spikes for a subset of neurons, sorted by mean activity in the displayed window. The population trace summarizes mean activity across the displayed neurons.

In [ ]:
# ── Neural activity ────────────────────────────────────────────────────────
import matplotlib.ticker as mticker

spikes_win = bundle["spikes"][:, VIZ_START : VIZ_START + VIZ_LEN]   # (N, W)
time_win   = time_valid[VIZ_START : VIZ_START + VIZ_LEN]

# Sort neurons by mean activity in the window, descending
sort_idx    = np.argsort(spikes_win.mean(axis=1))[::-1]
N_SHOW      = min(100, bundle["N"])
raster_idx  = sort_idx[:N_SHOW]
raster_data = spikes_win[raster_idx, :]  # (N_SHOW, W)

# ── Figure setup ───────────────────────────────────────────────────────────
CM    = 1 / 2.54
FSIZE = 7
BLACK = "#1a1a1a"

fig, axes = plt.subplots(
    3, 1, figsize=(14 * CM, 9 * CM),
    gridspec_kw={"height_ratios": [3, 1, 2]},
)
fig.subplots_adjust(left=0.12, right=0.96, top=0.93, bottom=0.13, hspace=0.55)


def _despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(labelsize=FSIZE - 1)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(4))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(4))


# ── Panel 1: Spike raster ──────────────────────────────────────────────────
ax_raster = axes[0]
for row_i in range(N_SHOW):
    active = np.where(raster_data[row_i] > 0)[0]
    if active.size:
        sizes = np.clip(raster_data[row_i, active] * 1.5, 0.5, 4)
        ax_raster.scatter(
            time_win[active], np.full(active.size, row_i),
            s=sizes, c=BLACK, linewidths=0, rasterized=True,
        )

ax_raster.set_xlim(time_win[0], time_win[-1])
ax_raster.set_ylim(-0.5, N_SHOW - 0.5)
ax_raster.invert_yaxis()
ax_raster.set_ylabel("Neurons", fontsize=FSIZE)
ax_raster.set_title("Spike raster", fontsize=FSIZE, pad=2)
ax_raster.tick_params(labelbottom=False, labelsize=FSIZE - 1)
ax_raster.spines["top"].set_visible(False)
ax_raster.spines["right"].set_visible(False)
ax_raster.yaxis.set_major_locator(mticker.MaxNLocator(4))

# ── Panel 2: Population activity trace ────────────────────────────────────
ax_pop = axes[1]
pop_mean   = raster_data.mean(axis=0)
pop_smooth = np.convolve(pop_mean, np.ones(5) / 5, mode="same")

ax_pop.plot(time_win, pop_smooth, color=BLACK, lw=0.8)
ax_pop.fill_between(time_win, pop_smooth, alpha=0.15, color=BLACK)
ax_pop.set_xlim(time_win[0], time_win[-1])
ax_pop.set_ylabel("Mean activity", fontsize=FSIZE)
ax_pop.tick_params(labelbottom=False)
_despine(ax_pop)

# ── Panel 3: Example single-neuron step traces ─────────────────────────────
ax_traces = axes[2]
N_TRACES    = 5
_pos        = np.linspace(0, N_SHOW - 1, N_TRACES, dtype=int)
example_idx = raster_idx[_pos]

_step = max(
    max(
        float(bundle["spikes"][ni, VIZ_START : VIZ_START + VIZ_LEN].max())
        for ni in example_idx
    ) + 1.0,
    2.0,
)
_t_span = float(time_win[-1] - time_win[0])

for k, ni in enumerate(example_idx):
    trace  = bundle["spikes"][ni, VIZ_START : VIZ_START + VIZ_LEN].astype(float)
    offset = k * _step
    ax_traces.step(time_win, trace + offset, where="post",
                   color=BLACK, lw=0.7, alpha=0.85)
    ax_traces.text(
        time_win[-1] + _t_span * 0.012, trace[-1] + offset,
        f"n{k+1}", fontsize=FSIZE - 1, va="center", color="#555555",
    )

ax_traces.set_xlim(time_win[0], time_win[-1] + _t_span * 0.08)
ax_traces.set_xlabel("Time (s)", fontsize=FSIZE)
ax_traces.set_ylabel("Example neurons (offset)", fontsize=FSIZE)
ax_traces.set_title("Example single-neuron activity", fontsize=FSIZE, pad=2)
ax_traces.set_yticks([])
_despine(ax_traces)
ax_traces.spines["left"].set_visible(False)

# ── Save ──────────────────────────────────────────────────────────────────
fig.savefig(SAVE_DIR / "neural_activity_raster.png", dpi=180)
plt.show()
print(f"Saved → {SAVE_DIR / 'neural_activity_raster.png'}")

---
## Behavioral variables

The preprocessed `.npz` files contain stimulus frames and neural spike trains only.
Behavioral time-series (position, velocity, lick, reward, head direction) are not
included in this release.

In [ ]:
# ── Behavioral variables ──────────────────────────────────────────────────────
# List every key present in the raw NPZ to confirm which data are available.
_all_keys = list(raw.keys())
print("Keys in raw NPZ file:", _all_keys)

_EXPECTED_NEURAL = {"frames", "spikes", "blackouts", "time", "cellarea"}
_BEHAVIORAL_QUERY = {"position", "velocity", "lick", "reward", "head_dir",
                     "speed", "heading", "eye", "pupil"}

_extra_keys   = set(_all_keys) - _EXPECTED_NEURAL
_behav_found  = _BEHAVIORAL_QUERY & set(_all_keys)

if _behav_found:
    print(f"\nBehavioral variables found in NPZ: {sorted(_behav_found)}")
else:
    print(
        "\nBehavioral variables (position, velocity, lick, reward, …) are not "
        "included in this release.\nThe NPZ files contain stimulus frames and "
        "neural spike trains only."
    )

if _extra_keys - _BEHAVIORAL_QUERY:
    print(f"\nOther unexpected keys: {sorted(_extra_keys - _BEHAVIORAL_QUERY)}")

---
## Modality alignment

The dataset is fully synchronized: each time index `t` pairs exactly one stimulus frame with one neural state vector. The cell below confirms this and prints a spot-check at one example timepoint.

In [ ]:
# ── Modality alignment ──────────────────────────────────────────────────────────
assert bundle["frames"].shape[0] == bundle["spikes"].shape[1] == len(time_valid), \
    "Mismatch: frames, spikes, and time arrays have different lengths."

print("Alignment check passed.")
print(f"  frames : {bundle['frames'].shape}   [T × C × H × W]")
print(f"  spikes : {bundle['spikes'].shape}   [N × T]")
print(f"  time   : {time_valid.shape}         [T]")

t_ex = VIZ_START + 50
print(f"\nSpot-check at index t = {t_ex}  (t = {time_valid[t_ex]:.2f} s):")
print(f"  Frame  : shape {bundle['frames'][t_ex].shape},  "
      f"mean pixel = {float(bundle['frames'][t_ex].mean()):.4f}")
print(f"  Spikes : {bundle['N']} neurons,  "
      f"mean = {bundle['spikes'][:, t_ex].mean():.3f},  "
      f"active = {int((bundle['spikes'][:, t_ex] > 0).sum())}")

---
## Croissant metadata

The cell below looks for a Croissant JSON-LD metadata file in the repository. Croissant is a machine-readable format for describing dataset fields, file structure, and license.

In [ ]:
# ── Croissant metadata ─────────────────────────────────────────────────────────
_croissant_candidates = [
    ROOT.parent / "mouse_vs_ai_data_preprocessed" / "croissant.json",
    ROOT / "metadata" / "croissant_preprocessed_track2.json",
    ROOT / "metadata" / "croissant.json",
    ROOT / "croissant.json",
]

croissant_path = next((p for p in _croissant_candidates if p.exists()), None)

if croissant_path:
    cj = json.loads(croissant_path.read_text())
    print(f"Loaded Croissant metadata from: {croissant_path}")
    print(f"  Name        : {cj.get('name', 'not available')}")
    print(f"  Description : {cj.get('description', 'not available')}")
    print(f"  License     : {cj.get('license', 'not available')}")
    resources = cj.get("distribution", cj.get("fileObjects", []))
    if resources:
        print(f"  Resources   : {len(resources)} file object(s)")
        for r in resources[:5]:
            name = r.get("name", r.get("@id", "?"))
            fmt  = r.get("encodingFormat", r.get("fileFormat", "?"))
            print(f"    {name}  [{fmt}]")
        if len(resources) > 5:
            print(f"    ... and {len(resources)-5} more")
else:
    print(
        "TODO: Croissant metadata for the preprocessed Track 2 release has not been\n"
        "created yet. A Croissant JSON-LD file should be added at:\n"
        "  metadata/croissant_preprocessed_track2.json\n\n"
        "It should describe the dataset fields (frames, spikes, blackouts, time, cellarea),\n"
        "file structure, neuron filter criteria (C0.005), sampling rate, frame dimensions,\n"
        "and license. It can be adapted from the raw-data Croissant file if one exists,\n"
        "or generated from scratch following https://github.com/mlcommons/croissant."
    )

---
## Dataset summary

In [ ]:
# ── Dataset summary (full 9-session release) ───────────────────────────────────────────────────
# Summarises all 9 sessions regardless of SESSION_SET.
# Uses hardcoded per-session values from the C0.005 preprocessing (2026-05-01).
# No sidecar files or extra file I/O required.

_DS_ALL_STEMS = _SESSION_STEMS["full_9session"]

# Per-session facts from C0.005-filtered release
# (N_kept = spikes.shape[0], T_full = blackouts.shape[1], N_orig from preprocessing records)
_DS_KNOWN = {
    "tigre569_p2s28_mousevAI_perturbs_preprocessed": {"N_kept": 1387,  "T_full":  7929, "N_orig":  7966},
    "tigre569_p2s35_mousevAI_perturbs_preprocessed": {"N_kept": 1965,  "T_full":  9747, "N_orig":  8793},
    "tigre569_p2s38_mousevAI_perturbs_preprocessed": {"N_kept": 2444,  "T_full": 12804, "N_orig":  8711},
    "tigre613_p2s21_mousevAI_perturbs_preprocessed": {"N_kept": 1265,  "T_full": 12266, "N_orig":  3993},
    "tigre613_p2s22_mousevAI_perturbs_preprocessed": {"N_kept": 1560,  "T_full": 15222, "N_orig":  4011},
    "tigre613_p2s23_mousevAI_perturbs_preprocessed": {"N_kept":  851,  "T_full": 11414, "N_orig":  3286},
    "tigre847_p2s5_mousevAI_perturbs_preprocessed":  {"N_kept": 1332,  "T_full": 12309, "N_orig":  6552},
    "tigre847_p2s8_mousevAI_perturbs_preprocessed":  {"N_kept": 1468,  "T_full": 12383, "N_orig":  7749},
    "tigre847_p2s9_mousevAI_perturbs_preprocessed":  {"N_kept": 1352,  "T_full":  9912, "N_orig":  5678},
}

_ds_frames_full = sum(v["T_full"]  for v in _DS_KNOWN.values())
_ds_kept        = sum(v["N_kept"]  for v in _DS_KNOWN.values())
_ds_orig        = sum(v["N_orig"]  for v in _DS_KNOWN.values())
_ds_n_sess      = len(_DS_ALL_STEMS)
_ds_minutes     = round(_ds_frames_full / 10 / 60, 1)
_ds_pct         = f"{100 * _ds_kept / _ds_orig:.0f}%"

_ds_ex_bundle = bundles[0]
_ds_ex_stem   = stems[0]
_ds_frame_shp = tuple(_ds_ex_bundle["frames"].shape[1:])  # (C, H, W)
_ds_spike_shp = _ds_ex_bundle["spikes"].shape              # (N, T)

print(f"{chr(8212)*62}")
print("  Dataset summary  (full 9-session release)")
print(f"{chr(8212)*62}")
print(f"  Sessions              : {_ds_n_sess}")
print(f"  Total recording time  : {_ds_minutes} min")
print(f"  Neurons retained      : {_ds_kept:,}")
print(f"  Original neuron pool  : {_ds_orig:,}")
print(f"  Retention rate        : {_ds_pct}")
print(f"  Stimulus frame shape  : {_ds_frame_shp}  [C x H x W]")
print(f"  Spike array (example) : {_ds_spike_shp}  [N x T]")
print(f"  Example session       : {_ds_ex_stem}")
print(f"{chr(8212)*62}")
print()

print(
    f"The preprocessed Track 2 release contains {_ds_n_sess} sessions "
    f"totalling {_ds_minutes} minutes of paired visual input and neural "
    f"activity. The release includes {_ds_kept:,} retained neurons "
    f"from an original pool of {_ds_orig:,} recorded neurons "
    "after quality-control filtering (C0.005: total spike count \u2265 100 "
    "AND R_visual \u2265 0.005)."
)